# Week 15: SSL-AASIST on the robustness sweep and the IndicVoices test speakers

Scores, with both SSL-AASIST checkpoints (seeds 1234 and 2025, the final recipe's data):

1. the robustness sweep (plan task A4): 1,122 clips under 15 post-processing transforms,
   rendered on the laptop (`src/data/robustness.py`; design fixed in
   `docs/results/robustness_v1.md` before any scoring);
2. the IndicVoices test speakers (plan task A2): 1,024 genuine spontaneous clips of 30
   speakers through the G.711 line, clip level, as HiACC was scored.

Clip-level scoring with the project's own scorer (`python -m src.baselines.aasist score`,
log-odds of genuine; called fake when below 0). Nothing is chosen here.

**How to run.** Input: `saikrishnareddy9/codemix-w15-scoring` (the rendered sweep, the
IndicVoices test clips and the two checkpoints). Settings: **GPU T4 x1** is enough,
Internet **On** (to clone the repository). Use **Save Version -> Save & Run All (Commit)**.

**Time.** About 30-40 minutes.

**Success.** The log shows `sweep found`, `iv_test found`, `checkpoints: 2`, then one line
per scored file; it ends with `RUN HEALTH: every step ran without error.` and a link to
`w15_scoring_results.zip`. Download the zip and put it in `D:\kaggle_realcall\`.

In [ ]:
import glob, json, os, subprocess, sys, time, traceback, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")              # saved output: only the zip goes here
TEMP = Path("/kaggle/temp")
try:
    TEMP.mkdir(parents=True, exist_ok=True)
except OSError:
    TEMP = Path("/tmp/w15")
    TEMP.mkdir(parents=True, exist_ok=True)
REPO = TEMP / "codemix-deepfake-detection"
OUT = TEMP / "scores"
ZIP_NAME = "w15_scoring_results.zip"
TRANSFORMS = ["none", "mp3_32k", "mp3_64k", "opus_12k", "aac_64k", "white_20db", "white_10db",
              "babble_20db", "babble_10db", "reverb_0.4s", "speed_0.95", "speed_1.05",
              "pitch_-1", "pitch_+1", "double_g711"]
FAILURES, DONE = [], []


def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        raise RuntimeError(f"exit {r.returncode}: {cmd}\n{r.stdout[-1500:]}\n{r.stderr[-3000:]}")
    return r.stdout


def finish():
    status = "every step ran without error." if not FAILURES else f"{len(FAILURES)} step(s) FAILED"
    summary = [f"RUN HEALTH: {status}", f"scored files: {len(DONE)}"] + [f"FAILED: {f}" for f in FAILURES]
    (OUT / "summary.txt").write_text("\n".join(summary) + "\n")
    with zipfile.ZipFile(WORK / ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in OUT.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(OUT))
    print("\n".join(summary))
    from IPython.display import FileLink, display
    os.chdir(WORK)
    print("\nCLICK THIS LINK TO DOWNLOAD THE RESULTS:")
    display(FileLink(ZIP_NAME))


try:
    OUT.mkdir(parents=True, exist_ok=True)
    sweep = sorted(glob.glob("/kaggle/input/**/sweep_none.csv", recursive=True))
    ivtest = sorted(glob.glob("/kaggle/input/**/iv_test_clips.csv", recursive=True))
    ckpts = sorted(glob.glob("/kaggle/input/**/ssl_aasist_final_s*_fp16.pt", recursive=True))
    if not sweep or not ivtest or len(ckpts) != 2:
        raise SystemExit("Attach saikrishnareddy9/codemix-w15-scoring (Add Input)")
    SWEEP_ROOT, IV_ROOT = Path(sweep[0]).parent, Path(ivtest[0]).parent
    print("sweep found:", SWEEP_ROOT, "| iv_test found:", IV_ROOT, "| checkpoints:", len(ckpts))
    if not REPO.exists():
        sh(f"git clone -q --depth 1 https://github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git {REPO}")
    import torch
    print("GPU:", torch.cuda.is_available())
    for ckpt in ckpts:
        seed = Path(ckpt).name.split("_s")[-1].split("_")[0]
        jobs = [(f"sweep_{t}", SWEEP_ROOT / f"sweep_{t}.csv", SWEEP_ROOT) for t in TRANSFORMS]
        jobs.append(("iv_test", IV_ROOT / "iv_test_clips.csv", IV_ROOT))
        for name, manifest, root in jobs:
            dest = OUT / f"ssl_s{seed}" / f"{name}_scores.csv"
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.exists():
                continue
            try:
                t0 = time.time()
                sh(f"cd {REPO} && {sys.executable} -m src.baselines.aasist score --model ssl --weights {ckpt} "
                   f"--manifest {manifest} --data-root {root} --device cuda --batch-size 32 "
                   f"--scores-out {dest} --out {dest.with_suffix('.json')}")
                DONE.append(str(dest))
                print(f"  ssl_s{seed} {name}: {time.time() - t0:.0f} s", flush=True)
            except Exception as e:
                FAILURES.append(f"ssl_s{seed} {name}: {str(e)[-800:]}")
                print("FAILED:", str(e)[-1500:], flush=True)
except BaseException:
    FAILURES.append(traceback.format_exc()[-1500:])
    print(traceback.format_exc())
finally:
    finish()
